In [ ]:
# ============================================================
# RADAR FEATURE EXTRACTION — Full Feature Set
# Features: Doppler frequency shift,
#           Micro-movement velocity,
#           Signal amplitude variation
# Source:   zoomed_RDM_cube from .mat files
#           shape → (range_bins, doppler_bins, frames)
# Frame → time conversion: t(s) = frame_index × Seconds_per_Frame
# Output: radar_features.csv saved to Google Drive
# ============================================================

import numpy as np
import pandas as pd
import scipy.io as sio
import os

# ── 1. Paths ──────────────────────────────────────────────────────────────
dataset_path = (
    '/content/501Project_Dataset/'
    'multimodal-synchronized-motion-capture-force-plate-and-radar-'
    'dataset-of-the-one-legged-stand-test-for-fall-risk-assessment-1.0'
)
output_path = '/content/drive/MyDrive/radar_features.csv'

# ── 2. Load OLST attempts ─────────────────────────────────────────────────
olst_df = pd.read_csv(f'{dataset_path}/Metadata/OLST_Attempts.csv')
olst_df['participant_id'] = olst_df['OLST_attempt_id'].str.slice(0, 2).astype(int)
olst_df['movement_code']  = olst_df['RADAR_capture'].str.split('_').str[1]
print(f'✓ Loaded OLST_Attempts.csv — {len(olst_df)} attempts')

# ── 3. Frame ↔ time helpers ───────────────────────────────────────────────
# Dataset documentation:
#   t_seconds = frame_index × Seconds_per_Frame
#   (RADAR_Start_Frame / RADAR_End_Frame are absolute indices within the
#    recording session; the .mat cube stores frames 0-indexed from the
#    start of the session, so cube frame 0 corresponds to RADAR_Start_Frame)
#
# To select frames [f_start, f_end] from the cube:
#   cube_idx = frame_number - RADAR_Start_Frame  (0-based into the cube)

def time_to_frame(t_seconds, spf, radar_start_frame):
    """Convert a timestamp (seconds) to an absolute radar frame number."""
    return int(round(t_seconds / spf)) + radar_start_frame


def frame_range_to_cube_slice(f_start, f_end, radar_start_frame, n_frames):
    """
    Convert absolute frame numbers to 0-based cube indices,
    clamped to valid range [0, n_frames).
    Returns (i_start, i_end) inclusive slice indices.
    """
    i_start = max(0, f_start - radar_start_frame)
    i_end   = min(n_frames - 1, f_end - radar_start_frame)
    return i_start, i_end


# ── 4. Feature functions ──────────────────────────────────────────────────

def doppler_shift_features(cube_window, doppler_axis_zoomed):
    """
    Doppler frequency shift features computed per radar frame.

    For each frame, the power-weighted mean Doppler frequency is the
    centroid of the Doppler spectrum collapsed across all range bins.
    This represents the dominant velocity component in the scene.

    Parameters
    ----------
    cube_window        : ndarray (range_bins, doppler_bins, n_frames)
    doppler_axis_zoomed: ndarray (doppler_bins,)  — Hz values for the
                         zoomed doppler indices

    Returns
    -------
    dict of scalar features
    """
    if cube_window.shape[2] < 2:
        return {}

    # Sum power across range bins → shape (doppler_bins, n_frames)
    doppler_power = cube_window.sum(axis=0)          # (doppler_bins, frames)

    # Power-weighted centroid per frame → dominant Doppler frequency [Hz]
    weights      = doppler_power + 1e-12             # avoid /0
    centroid     = (doppler_axis_zoomed[:, None] * weights).sum(axis=0) / weights.sum(axis=0)
    # centroid shape: (n_frames,)

    return {
        'doppler_centroid_mean': centroid.mean(),
        'doppler_centroid_std':  centroid.std(),
        'doppler_centroid_max':  np.abs(centroid).max(),
        'doppler_centroid_range': centroid.max() - centroid.min(),
    }


def micro_movement_velocity_features(cube_window, doppler_axis_zoomed, spf):
    """
    Micro-movement velocity features derived from frame-to-frame
    changes in Doppler centroid.

    The instantaneous velocity estimate (m/s) uses the standard
    Doppler-to-velocity relationship:
        v = (f_doppler × c) / (2 × f_carrier)
    where for a 24 GHz radar:
        c / (2 × f_carrier) ≈ 0.00625 m·s⁻¹·Hz⁻¹

    Parameters
    ----------
    cube_window        : ndarray (range_bins, doppler_bins, n_frames)
    doppler_axis_zoomed: ndarray (doppler_bins,)
    spf                : float  — seconds per frame (for acceleration)

    Returns
    -------
    dict of scalar features
    """
    if cube_window.shape[2] < 3:
        return {}

    DOPPLER_TO_VEL = 0.00625   # m/s per Hz  (c / 2f_c, 24 GHz radar)

    doppler_power = cube_window.sum(axis=0)
    weights       = doppler_power + 1e-12
    centroid      = (doppler_axis_zoomed[:, None] * weights).sum(axis=0) / weights.sum(axis=0)

    velocity      = centroid * DOPPLER_TO_VEL            # (n_frames,) m/s
    vel_abs       = np.abs(velocity)
    vel_diff      = np.diff(velocity)                    # frame-to-frame change
    acceleration  = vel_diff / spf                       # m/s²

    return {
        'micro_vel_mean':         vel_abs.mean(),
        'micro_vel_std':          vel_abs.std(),
        'micro_vel_max':          vel_abs.max(),
        'micro_vel_energy':       (velocity ** 2).mean(),   # mean kinetic proxy
        'micro_acc_mean':         np.abs(acceleration).mean(),
        'micro_acc_std':          np.abs(acceleration).std(),
        'micro_acc_max':          np.abs(acceleration).max(),
    }


def signal_amplitude_features(cube_window):
    """
    Signal amplitude variation features from the RDM cube.

    Amplitude (power) is integrated across all range-Doppler bins
    per frame, giving a scalar total-power time series.  Statistics
    on this series capture how much the overall radar return fluctuates
    — a proxy for body movement intensity.

    Parameters
    ----------
    cube_window : ndarray (range_bins, doppler_bins, n_frames)

    Returns
    -------
    dict of scalar features
    """
    if cube_window.shape[2] < 2:
        return {}

    # Total power per frame: sum over all range and Doppler bins
    total_power = cube_window.sum(axis=(0, 1))          # (n_frames,)

    # Peak power per frame: max cell in each frame
    peak_power  = cube_window.max(axis=(0, 1))          # (n_frames,)

    # Frame-to-frame amplitude change (temporal variation)
    power_diff  = np.diff(total_power)

    return {
        'amp_total_power_mean':  total_power.mean(),
        'amp_total_power_std':   total_power.std(),
        'amp_total_power_range': total_power.max() - total_power.min(),
        'amp_peak_power_mean':   peak_power.mean(),
        'amp_peak_power_std':    peak_power.std(),
        'amp_variation_mean':    np.abs(power_diff).mean(),   # mean |ΔP|
        'amp_variation_std':     np.abs(power_diff).std(),
        'amp_variation_max':     np.abs(power_diff).max(),
    }


def extract_all_radar_features(cube_window, doppler_axis_zoomed,
                                spf, stance, lifted,
                                attempt_id, participant_id, label):
    """Combine all radar feature groups into one row dict."""
    if cube_window.shape[2] < 3:
        return None

    feats = {
        'OLST_attempt_id': attempt_id,
        'participant_id':  participant_id,
        'stance_leg':      stance,
        'lifted_leg':      lifted,
        'label':           label,
    }

    feats.update(doppler_shift_features(cube_window, doppler_axis_zoomed))
    feats.update(micro_movement_velocity_features(cube_window, doppler_axis_zoomed, spf))
    feats.update(signal_amplitude_features(cube_window))

    return feats


# ── 5. Main extraction loop ───────────────────────────────────────────────
all_features  = []
skipped       = 0
loaded_mat    = {}          # cache: radar_capture → mat dict

for idx, row in olst_df.iterrows():

    if (idx + 1) % 100 == 0:
        print(f'  Processing {idx+1}/{len(olst_df)}...')

    attempt_id      = row['OLST_attempt_id']
    participant_id  = row['participant_id']
    radar_capture   = row['RADAR_capture']
    movement_code   = row['movement_code']

    # ── Timing columns (seconds) ──────────────────────────────────────────
    t_foot_up  = row['t_foot_up']
    t_stable   = row['t_stable']
    t_break    = row['t_break']
    t_end      = row['t_end']

    # ── Frame columns (pre-computed in CSV) ───────────────────────────────
    # Prefer the pre-computed frame columns when available; fall back to
    # converting from time using Seconds_per_Frame.
    frame_foot_up = row['frame_foot_up']
    frame_stable  = row['frame_stable']
    frame_break   = row['frame_break']
    frame_end     = row['frame_end']

    radar_start_frame = row['RADAR_Start_Frame']
    radar_end_frame   = row['RADAR_End_Frame']
    spf               = row['Seconds_per_Frame']   # seconds per radar frame

    stance = movement_code[-1]
    lifted = 'R' if stance == 'L' else 'L'

    # ── Load .mat file (with caching) ─────────────────────────────────────
    if radar_capture not in loaded_mat:
        # FIX: Corrected path for radar data, remove participant_id subfolder and 'Raw'/'Radar' from path
        mat_file = (
            f'{dataset_path}/Processed/Radar_RDMs/{radar_capture}.mat'
        )
        if not os.path.exists(mat_file):
            skipped += 1
            continue
        try:
            loaded_mat[radar_capture] = sio.loadmat(
                mat_file, struct_as_record=False, squeeze_me=True
            )
        except Exception:
            skipped += 1
            continue

    mat           = loaded_mat[radar_capture]
    cube          = mat['zoomed_RDM_cube']          # (range_bins, doppler_bins, total_frames)
    doppler_idx   = mat['doppler_idx']              # indices into full 512-bin doppler axis
    doppler_axis  = mat['doppler_axis']             # full 512-bin axis [Hz]
    doppler_axis_zoomed = doppler_axis[doppler_idx] # (doppler_bins,) — zoomed axis

    n_frames = cube.shape[2]

    # ── Helper: resolve absolute frame number ─────────────────────────────
    def resolve_frame(frame_col_val, t_val):
        """Use pre-computed frame if valid, else convert from time."""
        if pd.notna(frame_col_val):
            return int(frame_col_val)
        if pd.notna(t_val):
            return time_to_frame(t_val, spf, radar_start_frame)
        return None

    f_foot_up = resolve_frame(frame_foot_up, t_foot_up)
    f_stable  = resolve_frame(frame_stable,  t_stable)
    f_break   = resolve_frame(frame_break,   t_break)
    f_end     = resolve_frame(frame_end,     t_end)

    # ── STABLE window: f_stable → f_break (or f_end) ─────────────────────
    if f_stable is not None:
        f_stop = f_break if f_break is not None else f_end
        if f_stop is not None:
            i_start, i_end = frame_range_to_cube_slice(
                f_stable, f_stop, radar_start_frame, n_frames
            )
            window = cube[:, :, i_start:i_end + 1]
            feat   = extract_all_radar_features(
                window, doppler_axis_zoomed, spf,
                stance, lifted, attempt_id, participant_id,
                label='STABLE'
            )
            if feat:
                all_features.append(feat)

    # ── UNSTABLE window: f_foot_up → f_end (when t_stable is NaN) ────────
    if pd.isna(t_stable):
        if f_foot_up is not None and f_end is not None:
            i_start, i_end = frame_range_to_cube_slice(
                f_foot_up, f_end, radar_start_frame, n_frames
            )
            window = cube[:, :, i_start:i_end + 1]
            feat   = extract_all_radar_features(
                window, doppler_axis_zoomed, spf,
                stance, lifted, attempt_id, participant_id,
                label='UNSTABLE'
            )
            if feat:
                all_features.append(feat)

# ── 6. Save output ────────────────────────────────────────────────────────
radar_features_df = pd.DataFrame(all_features)

print(f'\n✓ Extraction complete!')
print(f'  Total rows  : {len(radar_features_df)}')

# Only attempt to access 'label' column if the DataFrame is not empty
if not radar_features_df.empty:
    print(f'  STABLE      : {(radar_features_df["label"] == "STABLE").sum()}')
    print(f'  UNSTABLE    : {(radar_features_df["label"] == "UNSTABLE").sum()}')
else:
    print(f'  STABLE      : 0')
    print(f'  UNSTABLE    : 0')

print(f'  Skipped     : {skipped}')
print(f'\nFeature columns ({len(radar_features_df.columns)}):')
print(radar_features_df.columns.tolist())

radar_features_df.to_csv(output_path, index=False)
print(f'\n✓ Saved to: {output_path}')